## **Pipeline de Alinhamento de Imagens** (via ECC)

Este notebook implementa a primeira etapa do pipeline de fotogrametria digital: o **alinhamento de imagens**, seguido pelo **recorte da região de interseção válida** para eliminar bordas pretas induzidas por transformações geométricas.

In [ ]:
# Bibliotecas
from utils.functions import load_images
import pandas as pd
import numpy as np
import cv2
import os

<br></br>

### **1. Fundamentação Teórica: ECC**

O algoritmo **ECC (*Enhanced Correlation Coefficient Maximization*)**, proposto por Evangelidis & Psarakis (2008), é uma abordagem direta para o registro de imagens. Diferente de métodos baseados em correlação cruzada padrão (NCC) ou soma de diferenças quadradas (SSD), o ECC é **invariante a mudanças afins na iluminação global**.

**Formulação Matemática:**
Dada uma imagem de referência $I_r(\mathbf{x})$ e uma imagem de entrada deformada $I_w(\mathbf{x}; \mathbf{p})$, o objetivo é encontrar o vetor de parâmetros ótimos $\hat{\mathbf{p}}$ que maximiza o coeficiente de correlação aprimorado $\rho(\mathbf{p})$:

$$\rho(\mathbf{p}) = \frac{\bar{\mathbf{i}}_r^T \bar{\mathbf{i}}_w(\mathbf{p})}{\|\bar{\mathbf{i}}_r\| \|\bar{\mathbf{i}}_w(\mathbf{p})\|}$$

Onde:
* $\bar{\mathbf{i}}_r$ é o vetor de intensidades com média zero da imagem de referência.
* $\bar{\mathbf{i}}_w(\mathbf{p})$ é o vetor de intensidades com média zero da imagem deformada sob a transformação parametrizada por $\mathbf{p}$.

> **Nota:** A otimização utiliza uma aproximação de Gauss-Newton modificada no espaço vetorial normalizado, garantindo convergência estável mesmo com variações moderadas de luminosidade.


<br></br>

### **2. Modelos de Transformação Geométrica**

A escolha da matriz de deformação depende da natureza das imagens. Por conta da estrutura deste experimento - câmera em trilho centrada no fóssil, utiliza-se MOTION_AFFINE, com mudança de escala.

| Modelo (`warp_mode`) | Graus de Liberdade (DoF) | Matriz de Transformação | Caso de Uso Recomendado |
| :--- | :---: | :---: | :--- |
| `MOTION_TRANSLATION` | 2 | $\begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \end{bmatrix}$ | Translação pura 2D (câmera ortográfica estática). |
| `MOTION_EUCLIDEAN` | 3 | $\begin{bmatrix} \cos\theta & -\sin\theta & t_x \\ \sin\theta & \cos\theta & t_y \end{bmatrix}$ | Rotação rígida + translação (sem escala). |
| `MOTION_AFFINE` | 6 | $\begin{bmatrix} a_{11} & a_{12} & t_x \\ a_{21} & a_{22} & t_y \end{bmatrix}$ | Cisalhamento (*shear*), escala não uniforme e rotação. |
| `MOTION_HOMOGRAPHY` | 8 | $\begin{bmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & 1 \end{bmatrix}$ | Perspectiva planar ou rotação pura de câmera com centro óptico fixo. |

<br></br>

### **3. Pipeline de Processamento**

A execução do alinhamento segue as seguintes etapas estruturadas:

1. **Pré-processamento:**
   * Conversão das imagens para escala de cinza (`cv2.COLOR_BGR2GRAY`).
   * *Opcional:* Aplicação de filtro Gaussiano para atenuação de ruído de alta frequência que possa degradar o cálculo dos gradientes espaciais.
2. **Critérios de Parada (`TermCriteria`):**
   * Limite máximo de iterações.
   * Tolerância de convergência $\epsilon$.
3. **Estimação da Matriz de Warp:**
   * Execução de `cv2.findTransformECC()`.
4. **Reamostragem Espacial:**
   * Aplicação de `cv2.warpAffine()` com interpolação linear.

<br></br>

### **4. Estratégia de Recorte**

A aplicação de transformações geométricas introduz regiões vazias (artefatos de borda com valor `0`) nas extremidades das imagens alinhadas.

**Algoritmo de Bounding Box de Interseção:**


Para garantir que todas as imagens processem exatamente os mesmos pixels nas etapas subsequentes de profundidade:
1. Cria-se uma máscara binária unitária $M_i$ para cada imagem alinhada.
2. Aplica-se a mesma transformação geométrica $\mathbf{W}_i$ sobre cada máscara.
3. Computa-se a interseção lógica de todas as máscaras:
   $$M_{\text{interseção}} = \bigcap_{i=1}^{N} \text{warp}(M_i, \mathbf{W}_i)$$
4. Extrai-se o maior retângulo interior ou o retângulo envolvente da interseção e aplica-se o *slice* uniforme em todo o conjunto de dados.

In [ ]:
# Função para alinhar o conjunto de imagens (ECC).
def align_images(images):
    """
    1. Alinha usando ECC.
    2. Rastreia as bordas pretas criadas pelo zoom/shift.
    3. Recorta o resultado final para eliminar áreas sem dados.
    """

    if not images: return [], []
    
    print(f"\nFase 1.2: Alinhamento ECC com Recorte ({len(images)} imagens)")
    
    # Referência - Ground Truth.
    ref_img = images[0]
    ref_gray = cv2.cvtColor(ref_img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    height, width = ref_img.shape[:2]
    
    aligned_images = [ref_img]
    scales = [1.0]
    acummulated_scales = scales.copy()
    common_mask = np.ones((height, width), dtype=np.uint8) * 255
    
    # Configuração ECC (Critérios de parada).
    warp_mode = cv2.MOTION_AFFINE
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 50, 1e-5)
    warp_matrix = np.eye(2, 3, dtype=np.float32)

    for i in range(1, len(images)):
        img_curr = images[i]
        curr_gray = cv2.cvtColor(img_curr, cv2.COLOR_BGR2GRAY).astype(np.float32)
        
        try:
            # Passo ECC
            step_matrix = np.eye(2, 3, dtype=np.float32)
            cc, step_matrix = cv2.findTransformECC(ref_gray, curr_gray, step_matrix, warp_mode, criteria)
            
            # Matriz Acumulada.
            step_3x3 = np.vstack([step_matrix, [0, 0, 1]])
            warp_3x3_accum = np.vstack([warp_matrix, [0,0,1]])
            combined = np.dot(step_3x3, warp_3x3_accum)
            warp_matrix = combined[:2, :]
            
            # Alinhamento da Imagem.
            img_aligned = cv2.warpAffine(img_curr, warp_matrix, (width, height), 
                                         flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP,
                                         borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0))
            
            aligned_images.append(img_aligned)
            
            # Atualização da Máscara de Validade.
            mask_curr = np.ones((height, width), dtype=np.uint8) * 255            
            mask_warped = cv2.warpAffine(mask_curr, warp_matrix, (width, height), 
                                         flags=cv2.INTER_NEAREST + cv2.WARP_INVERSE_MAP,
                                         borderMode=cv2.BORDER_CONSTANT, borderValue=0)
            
            common_mask = cv2.bitwise_and(common_mask, mask_warped)
            
            # Cálculo de Escalas.
            s_x = np.sqrt(warp_matrix[0,0]**2 + warp_matrix[1,0]**2)
            scales.append(s_x)
            acummulated_scales.append(acummulated_scales[-1]*s_x)
            
            print(f">> Img {i} Alinhada (ECC). Escala: {s_x:.4f}")

            # Nova Referência.
            ref_gray = curr_gray
            
        except cv2.error:
            print(f"[OS] Falha ECC na img {i}. Usando anterior.")
            aligned_images.append(aligned_images[-1])
            scales.append(scales[-1])
            acummulated_scales.append(acummulated_scales[-1]**2)
            
    # Corte
    contours, _ = cv2.findContours(common_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        x, y, w, h = cv2.boundingRect(contours[0])
        print(f"\n[OS] Área válida detectada: \n>> X: {x}, Y: {y}, Largura: {w}, Altura: {h}")
        print(f"[OS] Pixels perdidos nas bordas: \n>> Horizontal: {width-w}, Vertical: {height-h}")
        
        final_images = []
        for img in aligned_images:
            # Recorta todas as imagens no mesmo retângulo seguro.
            cropped = img[y:y+h, x:x+w]
            final_images.append(cropped)
            
        return final_images, scales, acummulated_scales
    else:
        print("[OS] Erro Crítico: A máscara comum ficou vazia! (Overlap insuficiente). Retornando originais.")
        return aligned_images, scales, acummulated_scales

In [ ]:
# Salva os Resultados.
def save_results(aligned_images, scales, acummulated_scales, output_folder, csv_path, resume_csv_path):
    """
    Salva as imagens processadas e um arquivo CSV com os dados de escala/zoom.
    """

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Pasta '{output_folder}' criada.")
    print(f"\n[OS] Salvando {len(aligned_images)} imagens e dados em '{output_folder}'...")
    
    with open(csv_path, mode='w', newline='') as file:
        for i, _ in enumerate(scales):
            filename = f"aligned_{i:04d}.jpg"
            filepath = os.path.join(output_folder, filename)

            cv2.imwrite(filepath, aligned_images[i])

    imgs_idx = np.arange(0, len(scales), 1)
    df = pd.DataFrame(columns=['imagens', 'escalas', 'escalas_acumuladas'])
    df['imagens'] = imgs_idx
    df['escalas'] = scales
    df['escalas_acumuladas'] = acummulated_scales
    resume = df.describe()

    df.to_csv(csv_path, index=False)
    resume.to_csv(resume_csv_path, index=True)
        
    print("[OS] Concluído! Todos os arquivos foram salvos.")
    return csv_path

In [ ]:
# Parâmetros.
images = load_images("../src/images/raw/shell")

In [ ]:
# Aplicar Alinhamento ECC nas Imagens Selecionadas.
aligned_images, scales, accumulated_scales = align_images(images)

In [ ]:
# Salvar os Resultados.
output_folder="../src/images/aligned/shell"
csv_path = "../results/scales/escalas.csv"
resume_csv_path = "../results/scales/resume.csv"

save_results(aligned_images, scales, accumulated_scales, output_folder, csv_path, resume_csv_path)